# 🏋️‍♂️ ATHX Leaderboard Web Scraping & Data Extraction

This notebook implements an automated web scraping pipeline to extract, parse, and structure official competition data from the **[ATHX Games Leaderboard](https://athxgames.com/team-leaderboards)**.

### Pipeline Overview:
1. **HTTP Requests & Parsing:** Scrape paginated leaderboard tables using `requests` and `BeautifulSoup`.
2. **Data Transformation:** Parse raw string metrics (weights in Kg, distances in Km, and time strings in MM:SS) into numeric formats.
3. **Data Export:** Clean and structure the dataset for exploratory data analysis (EDA) and performance simulation.

In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Base configuration and headers for scraping
BASE_URL = "https://athxgames.com/team-leaderboards"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("✅ Libraries imported and setup complete.")

✅ Libraries imported and setup complete.


## 1. Web Scraping Execution

This function iterates through all available pages for a targeted event configuration, extracting key fields for every competing team:
* **Rank & Team Name**
* **Zone 1 (Strength):** Lifted weights (Kg)
* **Zone 2 (Endurance):** Covered distance (Km)
* **Zone 3 (MetCon X):** Time taken (MM:SS)
* **Total Points**

In [4]:
def scrape_athx_leaderboard(target_url, total_pages=20):
    """
    Scrapes team leaderboard data across multiple pages.
    """
    scraped_data = []

    for page in range(1, total_pages + 1):
        # Adjust page URL parameter if required by target site structure
        page_url = f"{target_url}?page={page}"
        print(f"Scraping Page {page}/{total_pages}...")

        try:
            response = requests.get(page_url, headers=HEADERS, timeout=10)
            if response.status_code != 200:
                print(f"⚠️ Page {page} failed with status code {response.status_code}")
                continue

            soup = BeautifulSoup(response.content, "html.parser")
            table = soup.find("table")

            if not table:
                print(f"⚠️ No table found on page {page}")
                continue

            rows = table.find_all("tr")[1:]  # Skip table header

            for row in rows:
                cols = [col.text.strip() for col in row.find_all(["td", "th"])]
                if len(cols) >= 8:
                    scraped_data.append({
                        "Rank": cols[0],
                        "Team": cols[1],
                        "Event": cols[2],
                        "Country": cols[3],
                        "Year": cols[4],
                        "Division": cols[5],
                        "Strength_Raw": cols[6],
                        "Endurance_Raw": cols[7],
                        "MetCon_Raw": cols[8] if len(cols) > 8 else None,
                        "Points": cols[-1]
                    })

            time.sleep(1)  # Respectful scraping delay

        except Exception as e:
            print(f"❌ Error scraping page {page}: {e}")

    df = pd.DataFrame(scraped_data)
    print(f"\n🎉 Successfully scraped {len(df)} total team entries.")
    return df

# Example Execution (adjust target URL based on category/event)
# df_raw = scrape_athx_leaderboard(BASE_URL, total_pages=20)

TARGET_URL = "https://athxgames.com/team-leaderboards"
df_raw = scrape_athx_leaderboard(TARGET_URL, total_pages=20)

## 2. Data Cleaning & Metric Normalization

The raw scraped text contains formatting characters (e.g., `"804KG"`, `"10.487KM"`, `"11:16"`). This step standardizes metrics into numerical representations (`float` / `int`) for analytical modeling.

In [5]:
def clean_athx_dataset(df):
    """
    Cleans raw strings and converts metrics into numeric formats.
    """
    df_clean = df.copy()

    # 1. Clean Rank and Points
    df_clean["Rank"] = pd.to_numeric(df_clean["Rank"], errors="coerce")
    df_clean["Points"] = pd.to_numeric(df_clean["Points"], errors="coerce")

    # 2. Extract Strength (Kg)
    df_clean["Strength_Kg"] = (
        df_clean["Strength_Raw"]
        .str.extract(r"\((\d+)KG\)")
        .astype(float)
    )

    # 3. Extract Endurance (Km)
    df_clean["Endurance_Km"] = (
        df_clean["Endurance_Raw"]
        .str.extract(r"\(([\d\.]+)KM\)")
        .astype(float)
    )

    # 4. Extract MetCon Time & Convert to Total Seconds
    def parse_time_to_seconds(time_str):
        try:
            if "(" in str(time_str) and ")" in str(time_str):
                time_val = str(time_str).split("(")[1].replace(")", "").strip()
                minutes, seconds = map(int, time_val.split(":"))
                return minutes * 60 + seconds
        except Exception:
            return None
        return None

    df_clean["MetCon_Seconds"] = df_clean["MetCon_Raw"].apply(parse_time_to_seconds)

    # Drop intermediate raw columns
    cols_to_keep = [
        "Rank", "Team", "Event", "Country", "Year", "Division",
        "Strength_Kg", "Endurance_Km", "MetCon_Seconds", "Points"
    ]

    return df_clean[[col for col in cols_to_keep if col in df_clean.columns]]

# Example usage:
df_clean = clean_athx_dataset(df_raw)
df_clean.head()

## 3. Dataset Export

Export the cleaned, structured dataset to a CSV file to be stored in the repository data folder for further analysis and race simulation.

In [6]:
# Save dataset to CSV
df_clean.to_csv("athx_results_cleaned.csv", index=False)
print("💾 Cleaned dataset ready for EDA and Race Simulator modeling.")

💾 Cleaned dataset ready for EDA and Race Simulator modeling.
